# 1. Flask
> Flask 是一个用 Python 编写的轻量级 Web 应用框架。它被归类为微框架（microframework），因为它不需要特定的工具或库，保持核心简单但可扩展。

1. 轻量级：核心功能简洁，易于学习和使用
2. 灵活性：高度可定制，可以根据需要添加扩展
3. Werkzeug 和 Jinja2：基于这两个强大的库构建
4. RESTful：支持 RESTful 请求调度
5. 模板引擎：内置 Jinja2 模板引擎
6. 调试支持：内置开发服务器和调试器

## 1. Hello World
编写 `.py` 文件独立测试，不要在笔记本中运行代码
> pip install flask

In [ ]:
from flask import Flask, jsonify

app = Flask(__name__)

@app.route('/')
def hello_world():
    person = {
        "name": "Claude",
        "age": 25
    }
    return jsonify(person)

if __name__ == '__main__':
    print("服务器启动...")
    app.run(debug=True, host='0.0.0.0', port=5000)

## 2. 接收参数
### 1. 代码

In [ ]:
from flask import Flask, request, jsonify
import json

app = Flask(__name__)

# 1. URL 路径参数 (Path Parameters)
@app.route('/user/<int:user_id>')
def get_user_by_id(user_id):
    return jsonify({
        "method": "GET",
        "user_id": user_id,
        "type": type(user_id).__name__,
        "message": f"获取用户 ID: {user_id}"
    })

@app.route('/user/<string:username>')
def get_user_by_name(username):
    return jsonify({
        "method": "GET",
        "username": username,
        "type": type(username).__name__,
        "message": f"获取用户名: {username}"
    })

# 支持多个路径参数
@app.route('/user/<int:user_id>/posts/<int:post_id>')
def get_user_post(user_id, post_id):
    return jsonify({
        "user_id": user_id,
        "post_id": post_id,
        "message": f"获取用户 {user_id} 的帖子 {post_id}"
    })

# 2. 查询参数 (Query Parameters)
@app.route('/search')
def search():
    # 获取单个查询参数
    keyword = request.args.get('keyword', '')
    page = request.args.get('page', 1, type=int)
    limit = request.args.get('limit', 10, type=int)
    
    # 获取所有查询参数
    all_args = request.args.to_dict()
    
    # 获取列表类型的查询参数 (如: ?tags=python&tags=flask)
    tags = request.args.getlist('tags')
    
    return jsonify({
        "keyword": keyword,
        "page": page,
        "limit": limit,
        "tags": tags,
        "all_args": all_args,
        "message": "查询参数演示"
    })

# 3. 表单数据 (Form Data)
@app.route('/form', methods=['POST'])
def handle_form():
    # 获取表单数据
    name = request.form.get('name')
    email = request.form.get('email')
    age = request.form.get('age', type=int)
    
    # 获取所有表单数据
    all_form_data = request.form.to_dict()
    
    # 获取列表类型的表单数据
    hobbies = request.form.getlist('hobbies')
    
    return jsonify({
        "name": name,
        "email": email,
        "age": age,
        "hobbies": hobbies,
        "all_form_data": all_form_data,
        "content_type": request.content_type,
        "message": "表单数据接收成功"
    })

# 4. JSON 数据
@app.route('/json', methods=['POST'])
def handle_json():
    try:
        # 获取 JSON 数据
        json_data = request.get_json()
        
        # 检查是否有 JSON 数据
        if json_data is None:
            return jsonify({"error": "没有 JSON 数据"}), 400
        
        # 获取特定字段
        name = json_data.get('name')
        age = json_data.get('age')
        
        return jsonify({
            "received_data": json_data,
            "name": name,
            "age": age,
            "content_type": request.content_type,
            "message": "JSON 数据接收成功"
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

# 5. 文件上传
@app.route('/upload', methods=['POST'])
def handle_file_upload():
    # 检查是否有文件
    if 'file' not in request.files:
        return jsonify({"error": "没有文件"}), 400
    
    file = request.files['file']
    
    # 检查文件是否为空
    if file.filename == '':
        return jsonify({"error": "没有选择文件"}), 400
    
    # 获取其他表单数据
    description = request.form.get('description', '')
    
    return jsonify({
        "filename": file.filename,
        "content_type": file.content_type,
        "description": description,
        "file_size": len(file.read()),
        "message": "文件上传成功"
    })

# 6. 请求头 (Headers)
@app.route('/headers')
def handle_headers():
    # 获取特定请求头
    user_agent = request.headers.get('User-Agent')
    authorization = request.headers.get('Authorization')
    content_type = request.headers.get('Content-Type')
    
    # 获取所有请求头
    all_headers = dict(request.headers)
    
    return jsonify({
        "user_agent": user_agent,
        "authorization": authorization,
        "content_type": content_type,
        "all_headers": all_headers,
        "message": "请求头获取成功"
    })

# 7. Cookie 数据
@app.route('/cookies')
def handle_cookies():
    # 获取特定 Cookie
    session_id = request.cookies.get('session_id')
    user_pref = request.cookies.get('user_preference')
    
    # 获取所有 Cookie
    all_cookies = request.cookies.to_dict()
    
    return jsonify({
        "session_id": session_id,
        "user_preference": user_pref,
        "all_cookies": all_cookies,
        "message": "Cookie 数据获取成功"
    })

# 8. 原始请求数据
@app.route('/raw', methods=['POST'])
def handle_raw_data():
    # 获取原始请求体数据
    raw_data = request.get_data()
    
    # 尝试解码为文本
    try:
        text_data = raw_data.decode('utf-8')
    except:
        text_data = "无法解码为文本"
    
    return jsonify({
        "raw_data_length": len(raw_data),
        "raw_data_preview": str(raw_data)[:100],
        "text_data": text_data,
        "content_type": request.content_type,
        "message": "原始数据接收成功"
    })

# 9. 综合示例 - 处理多种参数类型
@app.route('/comprehensive/<int:id>', methods=['GET', 'POST', 'PUT', 'DELETE'])
def comprehensive_handler(id):
    result = {
        "method": request.method,
        "path_parameter": {"id": id},
        "query_parameters": request.args.to_dict(),
        "headers": dict(request.headers),
        "cookies": request.cookies.to_dict(),
    }
    
    # 根据请求方法处理不同类型的数据
    if request.method in ['POST', 'PUT']:
        if request.is_json:
            result["json_data"] = request.get_json()
        elif request.form:
            result["form_data"] = request.form.to_dict()
            result["files"] = [f.filename for f in request.files.values()]
        else:
            result["raw_data"] = request.get_data().decode('utf-8', errors='ignore')[:100]
    
    return jsonify(result)

# 10. 请求信息总览
@app.route('/request-info', methods=['GET', 'POST', 'PUT', 'DELETE'])
def request_info():
    return jsonify({
        "method": request.method,
        "url": request.url,
        "base_url": request.base_url,
        "path": request.path,
        "query_string": request.query_string.decode(),
        "remote_addr": request.remote_addr,
        "user_agent": request.user_agent.string,
        "content_type": request.content_type,
        "content_length": request.content_length,
        "is_json": request.is_json,
        "is_secure": request.is_secure,
        "scheme": request.scheme,
        "host": request.host,
    })

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)


### 2. 测试
#### 1. 测试路径参数
```sh
curl http://localhost:5000/user/123
curl http://localhost:5000/user/john
curl http://localhost:5000/user/123/posts/456
```
#### 2. 测试查询参数
```sh
curl "http://localhost:5000/search?keyword=python&page=2&limit=20&tags=web&tags=api"
```
#### 3. 测试表单数据
```sh
curl -X POST http://localhost:5000/form -d "name=张三&email=zhang@example.com&age=25&hobbies=编程&hobbies=读书"
```
#### 4. 测试 JSON 数据
```sh
curl -X POST http://localhost:5000/json \
  -H "Content-Type: application/json" \
  -d '{"name": "李四", "age": 30, "city": "北京"}'
```
#### 5. 测试文件上传
```sh
curl -X POST http://localhost:5000/upload \
  -F "file=@test.txt" \
  -F "description=测试文件"
```
#### 6. 测试请求头
```sh
curl http://localhost:5000/headers \
  -H "Authorization: Bearer token123" \
  -H "Custom-Header: custom-value"
```
#### 7. 测试 Cookie
```sh
curl http://localhost:5000/cookies \
  -H "Cookie: session_id=abc123; user_preference=dark_mode"
```
#### 8. 测试原始数据
```sh
curl -X POST http://localhost:5000/raw \
  -H "Content-Type: text/plain" \
  -d "这是原始文本数据"
```
#### 9. 综合测试
```sh
curl -X POST "http://localhost:5000/comprehensive/999?search=test&sort=name" \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer token" \
  -d '{"action": "update", "data": {"name": "测试"}}'
```

## 3. 封装大模型功能

In [ ]:
from transformers import pipeline

# 提前创建文本分类pipeline
classifier = pipeline("text-classification",
                     model='distilbert/distilbert-base-uncased-finetuned-sst-2-english',
                     top_k=None
                     )


def emotion_inference(text):

    # 分析文本情感
    result = classifier(text)
    return result


@app.route('/emotion')
def emotion():
    # 获取单个查询参数
    keyword = request.args.get('text', '')
    
    res = emotion_inference(keyword)
    
    return jsonify(res)

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)